Notebook illustrating the STA-LTA mining method of the paper. It is recommended to work through the if and dtw notebooks first.

We only consider data from ILL18 for 2018. 

Note: Results may differ from the paper since we are only using data from 2018 and working with the final catalog only.

In [27]:
import os
import numpy as np
import obspy
import pandas as pd

from seismicif.datamod.loading_utils import (
    preproc_flow_annotations,
    find_date_in_strings,
    remove_duplicate_traces,
    remove_overlaps,
    extract_split_flows,
)

from seismicif.datamod.preproc_utils import preproc_stream
from seismicif.sta_lta import slta_compute_iou, slta_detections_from_paths
from seismicif.metrics import (
    iou,
    est_thresholds,
    compute_statistics,
    extract_valid_segments,
)
from datetime import timedelta
from tqdm import tqdm
from itertools import product

In [12]:
folder = "../data/XP/2018/ILL18/EHZ.D/"
stream_paths = [folder + f for f in os.listdir(folder) if not f.startswith("._")]
stream_paths = np.array(sorted(stream_paths, key=lambda f: int(f.rsplit(".", 1)[-1])))

flows = preproc_flow_annotations(
    pd.read_csv("../catalogs/XP/flow_catalog.csv", index_col=0)
)
lower_conf_flows, high_conf_flows, all_flows = extract_split_flows(
    flows, "ILL18", 2018, 2018
)

As in the case of the IF trigger, we calibrate the STA-LTA trigger by focusing only on days with catalog segments.

Note. As far as is feasible (i.e. available data and no breakages) we append the seismic waveform of the previous day so that the STA-LTA can immediately compute its characteristic function.  

In [13]:
dates = []
for i in range(all_flows.shape[0]):
    dates.append(all_flows["start"].iloc[i].date)
    dates.append(all_flows["stop"].iloc[i].date)

dates = np.array(dates)
dates = np.unique(dates)

prev_days = []

for dt in dates:
    prev_days.append(dt - timedelta(days=1))

prev_days = np.array(prev_days)

st = []

for i in tqdm(range(dates.shape[0])):
    flow_st = obspy.read(find_date_in_strings(stream_paths, dates[i])[0])
    flow_st = remove_duplicate_traces(flow_st)
    flow_st = remove_overlaps(flow_st)
    preproc_stream(flow_st)

    prev_day_st = obspy.read(find_date_in_strings(stream_paths, prev_days[i])[0])
    prev_day_st = remove_duplicate_traces(prev_day_st)
    prev_day_st = remove_overlaps(prev_day_st)
    preproc_stream(prev_day_st)

    st.append({"res_tr": prev_day_st[-1], "st": flow_st})

100%|██████████| 7/7 [00:06<00:00,  1.14it/s]


Now we try to capture all the catalog segments as well as possible with the trigger parameters. 

Since we could not find a single grid that works well, we opted for local grid searches until no improvement in the IoU can be obtained.

In [17]:
# initilaize grids
sampling_rate = 100
lw_grid = 5000 * np.power(2.0, np.arange(-5, 6))
sw_grid = 0.003125 * np.power(2.0, np.arange(11))
onset_grid = 0.1875 * np.power(2.0, np.arange(11))
offset_grid = 0.00390625 * np.power(2.0, np.arange(11))

In [24]:
# iou_table is a bit artificial, but the grid search never yielded a parameter on the boundary.
try:
    iou_table = np.load("../output/XP/sta_lta/iou_grids/ILL18_illustation.npy")

except (FileNotFoundError, OSError):
    iou_table = np.zeros([11, 11, 11, 11])
    iou_table[:] = np.nan
    ic, jc, kc, lc = 5, 5, 5, 5
    stop = False

    while not stop:
        print("Performing Local Search")
        stop = True

        i_range = range(max(0, ic - 1), min(10, ic + 2))
        j_range = range(max(0, jc - 1), min(10, jc + 2))
        k_range = range(max(0, kc - 1), min(10, kc + 2))
        l_range = range(max(0, lc - 1), min(10, lc + 2))

        for i, j, k, l in tqdm(
            product(i_range, j_range, k_range, l_range),
            total=len(i_range) * len(j_range) * len(k_range) * len(l_range),
        ):
            if not np.isnan(iou_table[i, j, k, l]):
                continue

            stop = False
            lw = int(sampling_rate * lw_grid[j])
            sw = int(sampling_rate * sw_grid[i] * lw_grid[j])
            onset_thres, offset_thres = onset_grid[k], offset_grid[l]

            iou_table[i, j, k, l] = slta_compute_iou(
                st, all_flows, sw, lw, onset_thres, offset_thres
            )

        # all the positions around the current one has been filled.
        if stop:
            break

        temp_table = iou_table.copy()
        temp_table[np.isnan(temp_table)] = 0

        # check if we found a better solution than the current...otherwise stop
        if np.max(temp_table) > iou_table[ic, jc, kc, lc]:
            stop = False
            ic, jc, kc, lc = np.unravel_index(np.argmax(temp_table), temp_table.shape)

        else:
            stop = True

        np.save(f"../output/XP/sta_lta/iou_grids/ILL18_illustation.npy", iou_table)

iou_table[np.isnan(iou_table)] = 0
iou_table[np.isnan(iou_table)] = 0
ic, jc, kc, lc = np.unravel_index(np.argmax(iou_table), iou_table.shape)

lw = int(sampling_rate * lw_grid[jc])
sw = int(sampling_rate * sw_grid[ic] * lw_grid[jc])
onset_thres, offset_thres = onset_grid[kc], offset_grid[lc]
print(f"max iou: {100 * iou_table[ic, jc, kc, lc]:.2f}%")
print(
    f"Best parameters: lw={lw}, sw={sw}, onset_thres={onset_thres}, offset_thres={offset_thres}"
)

max iou: 45.09%
Best parameters: lw=4000000, sw=200000, onset_thres=12.0, offset_thres=0.5


In [25]:
# extract segments
sta_lta_segments = slta_detections_from_paths(
    stream_paths, sw, lw, onset_thres, offset_thres
)

100%|██████████| 181/181 [09:17<00:00,  3.08s/it]


When calibrating the minimum detection length and score thresholds we compare (a) only detections overlapping with a high confidence segment or no lower-confidence segments with (b) high confidence segments.

In [29]:
valid_segments = extract_valid_segments(
    sta_lta_segments, lower_conf_flows, high_conf_flows
)
iou_val, min_len, score_thres = est_thresholds(valid_segments, high_conf_flows)
print(f"iou_val={100 * iou_val:.2f}, min_len={min_len}, score_thres={score_thres}")

iou_val=28.45, min_len=3654.07, score_thres=14.276362528708537


In [30]:
detections = sta_lta_segments.copy()
start_times = np.array([pd.to_datetime(str(i)) for i in detections["start"]])
end_times = np.array([pd.to_datetime(str(i)) for i in detections["stop"]])
det_lens = np.array([i.total_seconds() for i in end_times - start_times])
detections = detections.assign(det_lens=det_lens)
detections = detections[
    (detections["scores"] > score_thres) & (detections["det_lens"] > min_len)
].reset_index(drop=True)
print(detections)

                         start                         stop     scores  \
0  2018-05-31T08:01:53.010003Z  2018-05-31T10:54:34.640003Z  19.890118   
1  2018-08-09T14:38:24.280002Z  2018-08-09T16:10:01.120002Z  19.738901   
2  2018-07-25T16:52:23.460019Z  2018-07-25T17:54:14.110019Z  19.671926   
3  2018-05-26T15:46:54.510003Z  2018-05-26T17:12:20.560003Z  16.444911   
4  2018-10-29T14:33:07.770000Z  2018-10-29T17:15:25.340000Z  15.805681   
5  2018-06-26T06:59:41.930003Z  2018-06-26T10:20:13.440003Z  14.799830   
6  2018-06-12T18:03:02.730003Z  2018-06-12T20:07:55.100003Z  14.310393   

   det_lens  
0  10361.63  
1   5496.84  
2   3710.65  
3   5126.05  
4   9737.57  
5  12031.51  
6   7492.37  
